In [46]:
import pandas as pd
import re
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt

In [47]:
df = pd.read_csv("cyberbullying_tweets.csv")
df

,tweet_text,cyberbullying_type
0,"In other words #katandandre, your food was cra...",not_cyberbullying
1,Why is #aussietv so white? #MKR #theblock #ImA...,not_cyberbullying
2,@XochitlSuckkks a classy whore? Or more red ve...,not_cyberbullying
3,"@Jason_Gio meh. :P thanks for the heads up, b...",not_cyberbullying
4,@RudhoeEnglish This is an ISIS account pretend...,not_cyberbullying
...,...,...
47687,"Black ppl aren't expected to do anything, depe...",ethnicity
47688,Turner did not withhold his disappointment. Tu...,ethnicity
47689,I swear to God. This dumb nigger bitch. I have...,ethnicity
47690,Yea fuck you RT @therealexel: IF YOURE A NIGGE...,ethnicity


In [48]:


df = df.applymap(lambda x: ' '.join(word for word in str(x).split() if not word.startswith('@'))) #Remove words starting with '@'
ampersand_pattern = re.compile(r'&\S*')
# Remove strings starting with '&' from all string columns
df = df.applymap(lambda x: ampersand_pattern.sub('', str(x)))
# Define a regular expression pattern to match web links
link_pattern = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+'
# Remove web links from all columns
df = df.applymap(lambda x: re.sub(link_pattern, '', str(x)))
www_pattern = re.compile(r'www\.[^\s]+') #Remove url's
df = df.applymap(lambda x: re.sub(www_pattern, '', str(x))) 
df = df.applymap(lambda x: ''.join(char for char in str(x) if not char.isdigit()))
df = df.applymap(lambda x: x.lower() if isinstance(x, str) else x)
df = df.applymap(lambda x: x.replace("rt", "") if isinstance(x, str) else x)
df = df.applymap(lambda x: re.sub(r'[^\w\s]', '', str(x)) if isinstance(x, str) else x)



C:\Users\adiks\AppData\Local\Temp\ipykernel_21452\547081142.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: ' '.join(word for word in str(x).split() if not word.startswith('@'))) #Remove words starting with '@'
C:\Users\adiks\AppData\Local\Temp\ipykernel_21452\547081142.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: ampersand_pattern.sub('', str(x)))
C:\Users\adiks\AppData\Local\Temp\ipykernel_21452\547081142.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(link_pattern, '', str(x)))
C:\Users\adiks\AppData\Local\Temp\ipykernel_21452\547081142.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: re.sub(www_pattern, '', str(x)))
C:\Users\adiks\AppData\Local\Temp\ipykernel_21452\547081142.py:11: FutureWarning: Data

In [49]:
custom_stopwords = set([
            "i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours", "yourself", "yourselves",
            "he", "him", "his", "himself", "she", "her", "hers", "herself", "it", "its", "itself", "they", "them", "their",
            "theirs", "themselves", "what", "which", "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was",
            "were", "be", "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an", "the", "and",
            "but", "if", "or", "because", "as", "until", "while", "of", "at", "by", "for", "with", "about", "against", "between",
            "into", "through", "during", "before", "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off",
            "over", "under", "again", "further", "then", "once", "here", "there", "when", "where", "why", "how", "all", "any", "both",
            "each", "few", "more", "most", "other", "some", "such", "no", "nor", "not", "only", "own", "same", "so", "than", "too", "very",
            "s", "t", "can", "will", "just", "don", "should", "now", "d", "ll", "m", "o", "re", "ve", "y", "ain", "aren", "couldn", "didn",
            "doesn", "hadn", "hasn", "haven", "isn", "ma", "mightn", "mustn", "needn", "shan", "shouldn", "wasn", "weren", "won", "wouldn"
        ])
df = df.applymap(lambda x: ' '.join(word for word in str(x).split() if word.lower() not in custom_stopwords))

C:\Users\adiks\AppData\Local\Temp\ipykernel_21452\2043402154.py:13: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: ' '.join(word for word in str(x).split() if word.lower() not in custom_stopwords))


In [50]:
custom_abbrevations = set([
            "lol", "omg", "lmao", "imo", "btw", "idk", "tbh", "rn", "thx", "brb", "asap", "aka", "irl", "smh", "tldr", "plz", "bc", "ily"
        ])
df = df.applymap(lambda x: ' '.join(word for word in str(x).split() if word.lower() not in custom_stopwords))

df

C:\Users\adiks\AppData\Local\Temp\ipykernel_21452\3147412107.py:4: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: ' '.join(word for word in str(x).split() if word.lower() not in custom_stopwords))


,tweet_text,cyberbullying_type
0,words katandandre food crapilicious mkr,not_cyberbullying
1,aussietv white mkr theblock imacelebrityau tod...,not_cyberbullying
2,classy whore red velvet cupcakes,not_cyberbullying
3,meh p thanks heads concerned another angry dud...,not_cyberbullying
4,isis account pretending kurdish account like i...,not_cyberbullying
...,...,...
47687,black ppl arent expected anything depended any...,ethnicity
47688,turner withhold disappointment turner called c...,ethnicity
47689,swear god dumb nigger bitch got bleach hair re...,ethnicity
47690,yea fuck youre nigger fucking unfollow fucking...,ethnicity


In [51]:
# Map categorical values in 'cyberbullying_type' to numerical values
label_mapping = {
    'religion': 1,
    'not_cyberbullying': 0,
    'age': 2,
    'ethnicity': 3,
    'gender': 4,
    'other_cyberbullying': 5
} 
df['cyberbullying_type'] = df['cyberbullying_type'].map(label_mapping)

In [ ]:
from nltk.tokenize import TweetTokenizer
from nltk.tokenize import TweetTokenizer
from nltk.stem.wordnet import WordNetLemmatizer

df = df.dropna(axis=0, subset=['tweet_text'])
df['tweet_text'] = df['tweet_text'].str.replace(r'\s*@\w+\s*', '', regex=True)                  #Removing @words
df['tweet_text'] = df['tweet_text'].str.replace(r'https?://\S+|http?://\S+', '', regex=True)    #Removing URLs
df['tweet_text'] = df['tweet_text'].str.lower()                                                 #Lowercasing all words
tweet_tokenizer = TweetTokenizer()
df['tokenized_tweet'] = df['tweet_text'].apply(lambda x: tweet_tokenizer.tokenize(x))                      #tokenized tweets //Time Consuming
lemmatizer = WordNetLemmatizer()                                                                #Lemmatize eg. doing = do,going=go //alternate:stemming
df['tokenized_tweet'] = df['tokenized_tweet'].apply(lambda x: [lemmatizer.lemmatize(word, pos='v') for word in x]) 

#Deleting entire row if no tweet text is present
mask = df['tweet_text'].notna() & df['tweet_text'].str.strip().astype(bool)
df = df[mask]
#df


,tweet_text,cyberbullying_type,tokenized_tweet
0,words katandandre food crapilicious mkr,0,"[word, katandandre, food, crapilicious, mkr]"
1,aussietv white mkr theblock imacelebrityau tod...,0,"[aussietv, white, mkr, theblock, imacelebritya..."
2,classy whore red velvet cupcakes,0,"[classy, whore, red, velvet, cupcakes]"
3,meh p thanks heads concerned another angry dud...,0,"[meh, p, thank, head, concern, another, angry,..."
4,isis account pretending kurdish account like i...,0,"[isis, account, pretend, kurdish, account, lik..."
...,...,...,...
47687,black ppl arent expected anything depended any...,3,"[black, ppl, arent, expect, anything, depend, ..."
47688,turner withhold disappointment turner called c...,3,"[turner, withhold, disappointment, turner, cal..."
47689,swear god dumb nigger bitch got bleach hair re...,3,"[swear, god, dumb, nigger, bitch, get, bleach,..."
47690,yea fuck youre nigger fucking unfollow fucking...,3,"[yea, fuck, youre, nigger, fuck, unfollow, fuc..."


In [ ]:
df.to_csv("final.csv", index=False) #Save the cleaned data to a new CSV file

In [ ]:
# label_map = {
#     'not_cyberbullying': 0,
#     'religion': 1,
#     'age': 2,
#     'ethnicity': 3,
#     'gender': 4,
#     'other_cyberbullying': 5
# }